## Exploratory SQL queries

Before building the marine heatwave detection logic, a few basic queries to get familiar with the data through SQL directly, rather than pandas.

In [1]:
import duckdb

In [2]:
con = duckdb.connect()

# Average SST by month, across all years, to see the seasonal cycle
result = con.execute("""
    SELECT MONTH(time) AS month,
           AVG(analysed_sst) AS avg_sst_kelvin,
           AVG(analysed_sst) - 273.15 AS avg_sst_celsius
    FROM '../data/processed/med_sst_2016_2026.parquet'
    GROUP BY month
    ORDER BY month
""").df()

print(result)

    month  avg_sst_kelvin  avg_sst_celsius
0       1      287.377574        14.227574
1       2      286.873459        13.723459
2       3      287.051288        13.901288
3       4      288.378220        15.228220
4       5      291.094656        17.944656
5       6      295.354534        22.204534
6       7      298.059271        24.909271
7       8      298.782748        25.632748
8       9      296.996113        23.846113
9      10      294.215715        21.065715
10     11      291.398052        18.248052
11     12      288.764087        15.614087


**Results**: Clear seasonal cycle, minimum in February (13.7 C) rather than January despite January having weaker sun, maximum in August (25.6 C) rather than June despite June having the longest days. This lag is consistent with ocean thermal inertia: the sea continues cooling into February from autumn heat loss, and keeps absorbing heat past the solstice before peaking in August.

The sharpest jump is May to June (+4.3 C), matching personal diving experience: shallow-water dives at Elba Island typically require a drysuit until mid to late May, consistent with the same late-spring warming pattern seen here. Predicted before running the query, based on years of scuba diving in the Tyrrhenian and Adriatic (where February and March are consistently the coldest months for diving).

## Checking year coverage before continuing

Before running further monthly queries, I check whether every month has the same number of years represented. The dataset runs from 2016-01-01 to 2026-06-21, so months after June likely have one fewer year of data than months January-June.

In [3]:
# Check how many distinct years of data exist per month
# to verify whether the 2016-2026 partial coverage skews the monthly averages
result_years = con.execute("""
    SELECT MONTH(time) AS month,
           COUNT(DISTINCT YEAR(time)) AS n_years,
           COUNT(DISTINCT time) AS n_days
    FROM '../data/processed/med_sst_2016_2026.parquet'
    GROUP BY month
    ORDER BY month
""").df()

print(result_years)

    month  n_years  n_days
0       1       11     341
1       2       11     311
2       3       11     341
3       4       11     330
4       5       11     341
5       6       11     321
6       7       10     310
7       8       10     310
8       9       10     300
9      10       10     310
10     11       10     300
11     12       10     310


In [4]:
# Compare January average with vs without 2026, to check if the extra year skews it
result_check = con.execute("""
    SELECT 
        AVG(CASE WHEN YEAR(time) < 2026 THEN analysed_sst END) - 273.15 AS avg_2016_2025,
        AVG(analysed_sst) - 273.15 AS avg_with_2026
    FROM '../data/processed/med_sst_2016_2026.parquet'
    WHERE MONTH(time) = 1
""").df()

print(result_check)

   avg_2016_2025  avg_with_2026
0      14.211833      14.227574


**Note on year coverage**: months January-June all have 11 years of data (2016-2026); for June specifically, only the day count within 2026 is partial (21 days instead of 30), not the year count. July-December have only 10 years (2016-2025), since the dataset ends mid-June 2026. Checked whether this asymmetry skews the monthly averages by comparing January with and without 2026: 14.21 C (2016-2025) vs 14.23 C (with 2026), a negligible 0.02 C difference. The imbalance is documented for transparency but does not meaningfully affect the monthly comparisons.

In [5]:
# Look at within-month variability across all available years (10-11 depending on month, see coverage note above):
# how much does temperature swing between the coldest and warmest day recorded in each month?
result = con.execute("""
    SELECT MONTH(time) AS month,
           AVG(analysed_sst) - 273.15 AS avg_celsius,
           MIN(analysed_sst) - 273.15 AS min_celsius,
           MAX(analysed_sst) - 273.15 AS max_celsius,
           MAX(analysed_sst) - MIN(analysed_sst) AS range_celsius
    FROM '../data/processed/med_sst_2016_2026.parquet'
    GROUP BY month
    ORDER BY month
""").df()

print(result)

    month  avg_celsius  min_celsius  max_celsius  range_celsius
0       1    14.227574     4.249994    18.009993          13.76
1       2    13.723459     4.529994    16.319994          11.79
2       3    13.901288     6.359994    16.759994          10.40
3       4    15.228220     9.849994    20.839993          10.99
4       5    17.944656    12.559994    26.379993          13.82
5       6    22.204534    14.959994    29.829993          14.87
6       7    24.909271    15.669994    30.659993          14.99
7       8    25.632748    16.279994    30.639993          14.36
8       9    23.846113    15.139994    29.359993          14.22
9      10    21.065715    13.679994    25.879993          12.20
10     11    18.248052     9.659994    23.829993          14.17
11     12    15.614087     4.569994    19.849993          15.28


**Results**: Range varies by month, June-July (14.87-14.99 C) were predicted to have the widest range and came close, but December is actually highest at 15.28 C, which was not anticipated. This range mixes temporal variability (cold day vs warm day within the same month) with geographic variability across the bounding box (the Gulf of Lion, exposed to the Mistral wind, can be substantially colder than the Tyrrhenian on the same day). December's wide range likely reflects this geographic spread, compounded by cold snaps from Mistral events against still-warm Tyrrhenian waters from summer thermal lag. The same lag mechanism explains why February is colder than January (observed earlier): the sea continues losing heat after the solstice, just as it continues absorbing heat past it in summer. Whether autumn storm activity also contributes to day-to-day variability within December is a plausible hypothesis but not something this aggregate query can confirm; it would require looking at within-month daily variability, not just the min/max across the whole 10-year, whole-grid sample.